# Ayan's `XGBDT SHAP` Values Analysis

## Purpose: 

{TODO}

In [6]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

import glob

import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
ayan_shap_folder = "/project/PlatigLab/data/collaborators/BWH/3_XGBDT_SHAP_data_2024_07/"
cell_lines = ["K562", "HepG2"]
distance_thresholds = list({file.split("/")[-1].split("-")[1] for file in glob.glob(f"{ayan_shap_folder}/**/*data.dat", recursive=True)})
best_performing_distance_threshold=100

shap_files = sorted(glob.glob(f"{ayan_shap_folder}/**/*data.dat", recursive=True))

actual_psi_column_name = "target"
predicted_psi_column_name = "psi_hat"

In [17]:
rbp_ppi=pd.read_excel("../../../inputs/RBP-RBP_PPI/lang_et_al_rec-y2h_screening_results.xlsx")
# rbp_ppi = rbp_ppi[(rbp_ppi["sumIS"]>=7.1) & (rbp_ppi["UniProt species A"]=="HUMAN") & (rbp_ppi["UniProt species B"]=="HUMAN")]

rbp_ppi = rbp_ppi[(rbp_ppi["sumIS"]>=7.1)]


rbp_ppi.shape
rbp_ppi.head()

rbp_ppi["Found in both orientations "].value_counts(normalize=True)

(2416, 57)

,Protein A,Protein B,UniProt species A,UniProt species B,UniProt accessions A,UniProt accessions B,UniProt IDs A,UniProt IDs B,Times detected (RIS > 0),avgIS,sumIS,Found in both orientations,H47_avgIS,H47_sumIS,H47_found in both orientations,H47_times detected (RIS > 0),"Biogrid_all direct evidence (Y2H, reconstituted complex, structure)",Biogrid_all,Hippie,HuRI,Known interaction,Protein A has known interactions,Protein B has known interactions,HPA nuclear A,HPA nuclear B,HPA cytoplasmic A,HPA cytoplasmic B,HPA main locations A,HPA additional locations A,HPA main locations B,HPA additional locations B,HPA shared locations,Youn et al. 2018 (PMID 29395067),NanoBRET,NanoBRET MBU,NanoBRET MBU std,ENCODE eCLIP data A,ENCODE eCLIP data B,ENCODE eCLIP binding sites A,ENCODE eCLIP binding sites B,Jaccard index,Cobinding prob p(A|B),Cobinding prob p(B|A),Cobinding prob p(A|B) (≤54 nt),Cobinding prob p(B|A) (≤54 nt),Close binding events (≤54 nt) A vs. B,Fraction of close binding events A vs. B,Fraction of close binding events in random data A vs. B,Ratio of fractions A vs. B,Resampling p-value A vs. B,Resampling Wilcoxon p-value A vs. B,Close binding events (≤54 nt) B vs. A,Fraction of close binding events B vs. A,Fraction of close binding events in random data B vs. A,Ratio of fractions B vs. A,Resampling p-value B vs. A,Resampling Wilcoxon p-value B vs. A
0,CTBP1,RBM14,HUMAN,MOUSE,Q13363,Q8C2Q3,CTBP1_HUMAN,RBM14_MOUSE,10,8.41,17.19,1,NaN,NaN,NaN,NaN,1,1,1,0,1,1,1,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nuclear speckles,NaN,NaN,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RBM14,CPSF6,MOUSE,MOUSE,Q8C2Q3,Q6NVF9,RBM14_MOUSE,CPSF6_MOUSE,11,13.98,16.03,0,NaN,NaN,NaN,NaN,0,0,0,0,0,1,0,1.0,1.0,0.0,0.0,Nuclear speckles,NaN,Nuclear speckles;Nucleoplasm,NaN,Nuclear speckles,0,NaN,NaN,NaN,0,1,0,791,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PTBP1,PTBP2,HUMAN,HUMAN,P26599,Q9UKA9,PTBP1_HUMAN,PTBP2_HUMAN,8,4.49,15.73,1,0.00,5.47,0.0,5.0,0,0,0,0,0,0,0,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nucleoplasm,NaN,Nucleoplasm,0,pos,13.34,0.94,1,0,16175,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SF1,EWSR1,HUMAN,MOUSE,Q15637,Q61545,SF01_HUMAN,EWS_MOUSE,3,3.29,15.50,0,7.17,11.66,0.0,3.0,1,1,1,0,1,1,1,1.0,1.0,0.0,0.0,Nucleoplasm,NaN,Nucleoplasm,Nucleoli,Nucleoplasm,0,NaN,NaN,NaN,0,1,0,8788,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SRSF11,SRPK2,HUMAN,HUMAN,Q05519,P78362,SRS11_HUMAN,SRPK2_HUMAN,10,9.52,14.79,1,2.32,5.06,0.0,5.0,0,1,1,1,1,1,1,1.0,1.0,0.0,1.0,Nuclear speckles,NaN,Cytosol;Nucleoplasm,NaN,NaN,0,NaN,NaN,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Found in both orientations 
0    0.93957
1    0.06043
Name: proportion, dtype: float64

## Despite larger distance thresholds having lower `R^2` performance, do they do better at predicting non-0/1 `PSI`? 

In [3]:

# for cell_line in cell_lines: 
    
#     for threshold in distance_thresholds: 
#         f"{cell_line} {threshold}"
        
#         files = [ file for file in shap_files if f"{cell_line}-{threshold}-" in file and "-train-" not in file ]
#         assert len(files)==2, [file.split("/")[-1] for file in files]        
        

#         plt.figure(dpi=100, figsize=(10,10))
        
        
#         df = [pd.read_csv(file, sep=",", usecols=[actual_psi_column_name, predicted_psi_column_name]) for file in files if "-train-" not in file]
        
#         df = pd.concat(df)
        
# #         sns.scatterplot(
# #             df, 
# #             x="target", 
# #             y="psi_hat", 
# #             size=0.00000000000001
        
# #         )

#         plt.hist2d(
        
#             df["target"].to_list(), 
#             df["psi_hat"].to_list(), 
#             bins=10
            
#         )
          
#         plt.show()

## Do `RBP PPI pairs` have similar `SHAP` in places where they bind together vs. don’t bind?

In [4]:
data_dict = {}

for cell_line in cell_lines: 
    data_dict[cell_line] = {}
    
    f"{cell_line}"

    files = [ file for file in shap_files if f"{cell_line}-{best_performing_distance_threshold}-" in file and "-train-" not in file ]
    assert len(files)==2, [file.split("/")[-1] for file in files]    

    df = [ pd.read_csv(file, sep=",") for file in files ]

    data_dict[cell_line] = pd.concat(df)

'K562'

'HepG2'

In [7]:
def get_interacting_rbp_columns(df, rbp_ppi, 

AARS_5_left  AATF_5_left  ABCF1_5_left  ADAT1_5_left  AGGF1_5_left  \
0            0            0             0             0             0   
1            0            0             0             0             0   
2            0            0             0             0             0   
3            0            0             0             0             0   
4            0            0             0             0             0   

   AKAP1_5_left  AKAP8L_5_left  APEX1_5_left  APOBEC3C_5_left  AQR_5_left  \
0             0              0             0                0           0   
1             0              0             0                0           0   
2             0              0             0                0           0   
3             0              0             0                0           0   
4             0              0             0                0           0   

   BUD13_5_left  CPEB4_5_left  CPSF6_5_left  CSTF2T_5_left  DDX1_5_left  \
0             0             0             0              0            0   
1             0             0             0              0            0   
2             0             0             0              0            0   
3             0             0             0              0            0   
4             0             0             0              0            0   

   DDX21_5_left  DDX24_5_left  DDX3X_5_left  DDX42_5_left  DDX43_5_left  \
0             0             0             0             0             0   
1             0             0             0             0             0   
2             0             0             0             0             0   
3             0             0             0             0             0   
4             0             1             0             0             0   

   DDX47_5_left  DDX51_5_left  DDX52_5_left  DDX55_5_left  DDX6_5_left  \
0             0             0             0             0            0   
1             0             0             0             0            0   
2             0             0             0             0            0   
3             0             0             0             0            0   
4             0             0             0             0            0   

   DGCR8_5_left  DHX30_5_left  DROSHA_5_left  EEF2_5_left  EFTUD2_5_left  \
0             0             0              0            0              0   
1             0             0              0            0              0   
2             0             0              0            0              0   
3             0             0              0            0              0   
4             0             0              0            0              0   

   EIF3G_5_left  EIF4E_5_left  EIF4G2_5_left  ELAC2_5_left  ELAVL1_5_left  \
0             0             0              0             0              0   
1             0             0              0             0              0   
2             0             0              0             0              0   
3             0             0              0             0              0   
4             0             0              0             0              0   

   EWSR1_5_left  EXOSC10_5_left  EXOSC5_5_left  FAM120A_5_left  \
0             0               0              0               0   
1             0               0              0               0   
2             0               0              0               0   
3             0               0              0               0   
4             0               0              0               0   

   FASTKD2_5_left  FMR1_5_left  FTO_5_left  FUS_5_left  FXR1_5_left  \
0               0            0           0           0            0   
1               0            0           0           0            0   
2               0            0           0           0            0   
3               0            0           0           0            0   
4               0            0           0           0            0   

   FXR2_5_

,AGGF1_5_left,AKAP1_5_left,AQR_5_left,BCCIP_5_left,BCLAF1_5_left,BUD13_5_left,CDC40_5_left,CSTF2_5_left,CSTF2T_5_left,DDX3X_5_left,DDX52_5_left,DDX55_5_left,DDX59_5_left,DDX6_5_left,DGCR8_5_left,DHX30_5_left,DKC1_5_left,DROSHA_5_left,EFTUD2_5_left,EIF3D_5_left,EIF3H_5_left,EXOSC5_5_left,FAM120A_5_left,FASTKD2_5_left,FKBP4_5_left,FTO_5_left,FUBP3_5_left,FUS_5_left,FXR2_5_left,G3BP1_5_left,GRSF1_5_left,GRWD1_5_left,GTF2F1_5_left,HLTF_5_left,HNRNPA1_5_left,HNRNPC_5_left,HNRNPK_5_left,HNRNPL_5_left,HNRNPM_5_left,HNRNPU_5_left,HNRNPUL1_5_left,IGF2BP1_5_left,IGF2BP3_5_left,ILF3_5_left,KHSRP_5_left,LARP4_5_left,LARP7_5_left,LIN28B_5_left,LSM11_5_left,MATR3_5_left,NCBP2_5_left,NIP7_5_left,NKRF_5_left,NOL12_5_left,NOLC1_5_left,NSUN2_5_left,PABPN1_5_left,PCBP1_5_left,PCBP2_5_left,POLR2G_5_left,PPIG_5_left,PRPF4_5_left,PRPF8_5_left,PTBP1_5_left,QKI_5_left,RBFOX2_5_left,RBM15_5_left,RBM22_5_left,RBM5_5_left,RPS3_5_left,SAFB_5_left,SDAD1_5_left,SF3A3_5_left,SF3B4_5_left,SFPQ_5_left,SLTM_5_left,SMNDC1_5_left,SND1_5_left,SRSF1_5_left,SRSF7_5_left,SRSF9_5_left,SSB_5_left,STAU2_5_left,SUB1_5_left,SUGP2_5_left,SUPV3L1_5_left,TAF15_5_left,TARDBP_5_left,TBRG4_5_left,TIA1_5_left,TIAL1_5_left,TRA2A_5_left,TROVE2_5_left,U2AF1_5_left,U2AF2_5_left,UCHL5_5_left,UPF1_5_left,UTP18_5_left,WDR43_5_left,XPO5_5_left,XRCC6_5_left,XRN2_5_left,YBX3_5_left,ZC3H11A_5_left,ZNF800_5_left,AGGF1_5_right,AKAP1_5_right,AQR_5_right,BCCIP_5_right,BCLAF1_5_right,BUD13_5_right,CDC40_5_right,CSTF2_5_right,CSTF2T_5_right,DDX3X_5_right,DDX52_5_right,DDX55_5_right,DDX59_5_right,DDX6_5_right,DGCR8_5_right,DHX30_5_right,DKC1_5_right,DROSHA_5_right,EFTUD2_5_right,EIF3D_5_right,EIF3H_5_right,EXOSC5_5_right,FAM120A_5_right,FASTKD2_5_right,FKBP4_5_right,FTO_5_right,FUBP3_5_right,FUS_5_right,FXR2_5_right,G3BP1_5_right,GRSF1_5_right,GRWD1_5_right,GTF2F1_5_right,HLTF_5_right,HNRNPA1_5_right,HNRNPC_5_right,HNRNPK_5_right,HNRNPL_5_right,HNRNPM_5_right,HNRNPU_5_right,HNRNPUL1_5_right,IGF2BP1_5_right,IGF2BP3_5_right,ILF3_5_right,KHSRP_5_right,LARP4_5_right,LARP7_5_right,LIN28B_5_right,LSM11_5_right,MATR3_5_right,NCBP2_5_right,NIP7_5_right,NKRF_5_right,NOL12_5_right,NOLC1_5_right,NSUN2_5_right,PABPN1_5_right,PCBP1_5_right,PCBP2_5_right,POLR2G_5_right,PPIG_5_right,PRPF4_5_right,PRPF8_5_right,PTBP1_5_right,QKI_5_right,RBFOX2_5_right,RBM15_5_right,RBM22_5_right,RBM5_5_right,RPS3_5_right,SAFB_5_right,SDAD1_5_right,SF3A3_5_right,SF3B4_5_right,SFPQ_5_right,SLTM_5_right,SMNDC1_5_right,SND1_5_right,SRSF1_5_right,SRSF7_5_right,SRSF9_5_right,SSB_5_right,STAU2_5_right,SUB1_5_right,SUGP2_5_right,SUPV3L1_5_right,TAF15_5_right,TARDBP_5_right,TBRG4_5_right,TIA1_5_right,TIAL1_5_right,TRA2A_5_right,TROVE2_5_right,U2AF1_5_right,U2AF2_5_right,UCHL5_5_right,UPF1_5_right,UTP18_5_right,WDR43_5_right,XPO5_5_right,XRCC6_5_right,XRN2_5_right,YBX3_5_right,ZC3H11A_5_right,ZNF800_5_right,AGGF1_center_left,AKAP1_center_left,AQR_center_left,BCCIP_center_left,BCLAF1_center_left,BUD13_center_left,CDC40_center_left,CSTF2_center_left,CSTF2T_center_left,DDX3X_center_left,DDX52_center_left,DDX55_center_left,DDX59_center_left,DDX6_center_left,DGCR8_center_left,DHX30_center_left,DKC1_center_left,DROSHA_center_left,EFTUD2_center_left,EIF3D_center_left,EIF3H_center_left,EXOSC5_center_left,FAM120A_center_left,FASTKD2_center_left,FKBP4_center_left,FTO_center_left,FUBP3_center_left,FUS_center_left,FXR2_center_left,G3BP1_center_left,GRSF1_center_left,GRWD1_center_left,GTF2F1_center_left,HLTF_center_left,HNRNPA1_center_left,HNRNPC_center_left,HNRNPK_center_left,HNRNPL_center_left,HNRNPM_center_left,HNRNPU_center_left,HNRNPUL1_center_left,IGF2BP1_center_left,IGF2BP3_center_left,ILF3_center_left,KHSRP_center_left,LARP4_center_left,LARP7_center_left,LIN28B_center_left,LSM11_center_left,MATR3_center_left,NCBP2_center_left,NIP7_center_left,NKRF_center_left,NOL12_center_left,NOLC1_center_left,NSUN2_center_left,PABPN1_center_left,PCBP1_center_left,PCBP2_center_left,POLR2G_center_left,PPIG_center_left,PRPF4_center_left,PRPF8_center_left,PTBP1